# Phase 5.4: Full 40 fold Leave-One-Subject-Out CV

Runs LOSO-CV on the winning Phase 5.1b architecture (SimpleEEGCNNv2).

**Methodology:**

For each of 40 subjects:
1. Hold that subject's ~60 segments out as test.
2. Use the other 39 subjects (~2340 segments) as training.
3. Compute class weights from those 39 subjects' labels.
4. Train SimpleEEGCNNv2 fresh for 12 epochs (no early stopping).
    - Why 12? Phase 5.1b found val_acc peaked at epoch 10-11 with the same architecture and data scale. 12 is a calibrated choice, not an arbitrary one.
5. Predict on the held-out subject; record metrics.

Aggregate across 40 folds.

In [2]:

import os
import json
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
 
from phase5_utils import (
    load_phase2_data, EEGDataset, evaluate_model, count_parameters,
    PHASE5_DIR, SEED,
)
 
torch.manual_seed(SEED)
np.random.seed(SEED)

# Winning Architecture

phase5.1b (SimpleEEGCNNv2): temporal → spatial → temporal → classify

In [3]:

class SimpleEEGCNNv2(nn.Module):

    def __init__(self, n_classes=2, n_temporal=4, n_spatial=8, dropout=0.6):
        super().__init__()
        self.temporal_conv = nn.Conv2d(1, n_temporal, kernel_size=(1, 25), padding=(0, 12), bias=False)

        self.bn1 = nn.BatchNorm2d(n_temporal)
        self.spatial_conv = nn.Conv2d(n_temporal, n_spatial, kernel_size=(32, 1), bias=False)

        self.bn2 = nn.BatchNorm2d(n_spatial)
        self.pool1 = nn.AvgPool2d(kernel_size=(1, 4))
        self.drop1 = nn.Dropout(dropout)
        self.temporal_conv2 = nn.Conv2d(n_spatial, n_spatial, kernel_size=(1, 13), padding=(0, 6), bias=False)
        
        self.bn3 = nn.BatchNorm2d(n_spatial)
        self.pool2 = nn.AvgPool2d(kernel_size=(1, 8))
        self.drop2 = nn.Dropout(dropout)
        self.classifier = nn.Linear(n_spatial * 20, n_classes)
 
    def forward(self, x):
        x = self.temporal_conv(x); x = self.bn1(x)
        x = self.spatial_conv(x); x = self.bn2(x); x = F.elu(x)
        x = self.pool1(x); x = self.drop1(x)
        x = self.temporal_conv2(x); x = self.bn3(x); x = F.elu(x)
        x = self.pool2(x); x = self.drop2(x)
        x = x.flatten(1); x = self.classifier(x)
        return x

# Per-Fold Training

Train for a fixed number of epochs (12). No Val/Early stop.

In [4]:
def train_one_fold(model, train_loader, criterion, n_epochs=12, lr=5e-4, weight_decay=1e-3, device="cpu"):
    # Returns final model state
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    for epoch in range(n_epochs):
        model.train()
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            optimizer.zero_grad()
            logits = model(X_batch)
            loss = criterion(logits, y_batch)
            loss.backward()
            optimizer.step()
    return model

# Main LOSO Loop

In [5]:

print("PHASE 5.4 — FULL 40-FOLD LEAVE-ONE-SUBJECT-OUT CROSS-VALIDATION")
print("Architecture: SimpleEEGCNNv2 (Phase 5.1b)")
print("=" * 70)
 
 
# Load data ----------------------------
print("\n[Setup] Loading data...")
X, y_binary, subjects = load_phase2_data(verbose=False)
unique_subjects = sorted(np.unique(subjects).tolist())
print(f"  X: {X.shape}, y: {y_binary.shape}, subjects: {len(unique_subjects)}")
 
 
# Run LOSO ----------------------------
device = "cpu"
HYPERPARAMS = dict(
    n_temporal=4, n_spatial=8, dropout=0.6,
    n_epochs=12, lr=5e-4, weight_decay=1e-3,
    batch_size=64,
)
print(f"\n[Setup] Hyperparameters: {HYPERPARAMS}")
print(f"\n[Setup] Beginning 40-fold LOSO...\n")
 
per_fold = []                # one dict per fold
all_preds_global = []        # for concatenated confusion matrix
all_labels_global = []
 
start_time = time.time()
for i, test_subj in enumerate(unique_subjects, start=1):
    fold_start = time.time()
 
    # Build masks
    test_mask = subjects == test_subj
    train_mask = ~test_mask
 
    # Loaders for this fold
    train_ds = EEGDataset(X[train_mask], y_binary[train_mask])
    test_ds = EEGDataset(X[test_mask],  y_binary[test_mask])
 
    g = torch.Generator(); g.manual_seed(SEED + i)   # different shuffle per fold
    train_loader = DataLoader(train_ds, batch_size=HYPERPARAMS["batch_size"], shuffle=True, generator=g)
    test_loader = DataLoader(test_ds, batch_size=HYPERPARAMS["batch_size"], shuffle=False)
 
    # Per-fold class weights (computed from THIS fold's training labels only)
    y_train_fold = y_binary[train_mask]
    counts = np.bincount(y_train_fold, minlength=2)
    weights = counts.sum() / (2 * counts)
    criterion = nn.CrossEntropyLoss(weight=torch.FloatTensor(weights))
 
    # Fresh model each fold (no transfer between folds)
    torch.manual_seed(SEED + i)              # deterministic init per fold
    model = SimpleEEGCNNv2(
        n_temporal=HYPERPARAMS["n_temporal"],
        n_spatial=HYPERPARAMS["n_spatial"],
        dropout=HYPERPARAMS["dropout"],
    ).to(device)
 
    # Train and evaluate
    model = train_one_fold(
        model, train_loader, criterion,
        n_epochs=HYPERPARAMS["n_epochs"],
        lr=HYPERPARAMS["lr"],
        weight_decay=HYPERPARAMS["weight_decay"],
        device=device,
    )
    results = evaluate_model(model, test_loader, device=device)
 
    fold_time = time.time() - fold_start
    n_test = int(test_mask.sum())
    n_pos_test = int(y_binary[test_mask].sum())
 
    per_fold.append({
        "fold": i,
        "test_subject": int(test_subj),
        "n_test_segments": n_test,
        "n_test_stress": n_pos_test,
        "accuracy": float(results["accuracy"]),
        "f1": float(results["f1"]),
        "f1_macro": float(results["f1_macro"]),
        "time_sec": float(fold_time),
        "tn": int(results["confusion_matrix"][0, 0]),
        "fp": int(results["confusion_matrix"][0, 1]),
        "fn": int(results["confusion_matrix"][1, 0]),
        "tp": int(results["confusion_matrix"][1, 1]),
    })
    all_preds_global.append(results["preds"])
    all_labels_global.append(results["labels"])
 
    # Running average for ETA
    avg_time = (time.time() - start_time) / i
    eta_remaining = avg_time * (len(unique_subjects) - i)
    print(f"  Fold {i:2d}/40 | test_subj={test_subj:2d} | "
          f"acc={results['accuracy']:.3f} | "
          f"f1_macro={results['f1_macro']:.3f} | "
          f"time={fold_time:4.1f}s | ETA {eta_remaining/60:4.1f}min")
 
total_time = time.time() - start_time
print(f"\nLOSO complete in {total_time:.1f}s ({total_time/60:.1f} min).")

PHASE 5.4 — FULL 40-FOLD LEAVE-ONE-SUBJECT-OUT CROSS-VALIDATION
Architecture: SimpleEEGCNNv2 (Phase 5.1b)

[Setup] Loading data...
  X: (2400, 32, 640), y: (2400,), subjects: 40

[Setup] Hyperparameters: {'n_temporal': 4, 'n_spatial': 8, 'dropout': 0.6, 'n_epochs': 12, 'lr': 0.0005, 'weight_decay': 0.001, 'batch_size': 64}

[Setup] Beginning 40-fold LOSO...

  Fold  1/40 | test_subj= 1 | acc=0.650 | f1_macro=0.627 | time=47.5s | ETA 30.9min
  Fold  2/40 | test_subj= 2 | acc=0.617 | f1_macro=0.591 | time=80.3s | ETA 40.5min
  Fold  3/40 | test_subj= 3 | acc=0.683 | f1_macro=0.656 | time=63.9s | ETA 39.4min
  Fold  4/40 | test_subj= 4 | acc=0.483 | f1_macro=0.482 | time=116.8s | ETA 46.3min
  Fold  5/40 | test_subj= 5 | acc=0.650 | f1_macro=0.642 | time=41.9s | ETA 40.9min
  Fold  6/40 | test_subj= 6 | acc=0.450 | f1_macro=0.437 | time=151.4s | ETA 47.4min
  Fold  7/40 | test_subj= 7 | acc=0.450 | f1_macro=0.450 | time=266.7s | ETA 60.4min
  Fold  8/40 | test_subj= 8 | acc=0.583 | f1_mac

# Aggregate Results

In [6]:

print("AGGREGATE LOSO RESULTS: Phase 5.4 (SimpleEEGCNNv2, 40-fold LOSO)")
print("=" * 70)

df = pd.DataFrame(per_fold)
accs = df["accuracy"].values
f1s = df["f1"].values
f1ms = df["f1_macro"].values

print(f"\nAccuracy  : {accs.mean():.4f} ± {accs.std():.4f}  "f"[min={accs.min():.3f}, median={np.median(accs):.3f}, max={accs.max():.3f}]")
print(f"F1 (pos)  : {f1s.mean():.4f} ± {f1s.std():.4f}")
print(f"F1 macro  : {f1ms.mean():.4f} ± {f1ms.std():.4f}")
print(f"\nSubjects above 0.587 (Phase 4B RF baseline): "f"{int((accs > 0.587).sum())}/40")
print(f"Subjects above 0.604 (majority baseline)    : "f"{int((accs > 0.604).sum())}/40")

# Global confusion matrix (concatenated predictions across all folds)
preds_all = np.concatenate(all_preds_global)
labels_all = np.concatenate(all_labels_global)
from sklearn.metrics import confusion_matrix
cm_global = confusion_matrix(labels_all, preds_all)
global_acc = (preds_all == labels_all).mean()
print(f"\nGlobal (concatenated) accuracy across all 2400 predictions: {global_acc:.4f}")
print(f"Global confusion matrix:\n{cm_global}")

AGGREGATE LOSO RESULTS: Phase 5.4 (SimpleEEGCNNv2, 40-fold LOSO)

Accuracy  : 0.5525 ± 0.1182  [min=0.300, median=0.575, max=0.750]
F1 (pos)  : 0.6018 ± 0.1689
F1 macro  : 0.5001 ± 0.1129

Subjects above 0.587 (Phase 4B RF baseline): 18/40
Subjects above 0.604 (majority baseline)    : 14/40

Global (concatenated) accuracy across all 2400 predictions: 0.5525
Global confusion matrix:
[[413 697]
 [377 913]]


# Save Outputs

In [7]:
out_dir = os.path.join(PHASE5_DIR, "phase5_4_loso")
os.makedirs(out_dir, exist_ok=True)
 
# Per-fold CSV ----------------------------------------
df.to_csv(os.path.join(out_dir, "per_fold_results.csv"), index=False)
 
#  Per-subject accuracy bar plot ----------------------------------------
fig, ax = plt.subplots(figsize=(14, 5))
order = df.sort_values("accuracy").reset_index(drop=True)
colors = ["#F15854" if a < 0.5 else "#5DA5DA" if a < 0.587 else "#60BD68"
          for a in order["accuracy"]]
bars = ax.bar(range(len(order)), order["accuracy"], color=colors)
ax.axhline(accs.mean(), color="black", linestyle="--", label=f"LOSO mean = {accs.mean():.3f}")
ax.axhline(0.587, color="purple", linestyle=":", label="Phase 4B RF LOSO = 0.587")
ax.axhline(0.604, color="gray", linestyle=":", label="Majority baseline = 0.604")
ax.axhline(0.5, color="red", linestyle=":", alpha=0.5, label="Chance = 0.500")
ax.set_xticks(range(len(order)))
ax.set_xticklabels(order["test_subject"].astype(int), rotation=0, fontsize=8)
ax.set_xlabel("Held-out subject (sorted by accuracy)")
ax.set_ylabel("Test accuracy")
ax.set_title("Phase 5.4 — LOSO accuracy per subject  "f"(mean={accs.mean():.3f}, std={accs.std():.3f})")
ax.legend(loc="lower right"); ax.grid(axis="y", alpha=0.3); ax.set_ylim(0, 1)
plt.tight_layout()
plt.savefig(os.path.join(out_dir, "per_subject_accuracy.png"), dpi=150)
plt.close()
 
# Distribution histogram ----------------------------------------
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.hist(accs, bins=15, color="#5DA5DA", edgecolor="black", alpha=0.85)
ax.axvline(accs.mean(), color="black", linestyle="--", label=f"Mean {accs.mean():.3f}")
ax.axvline(0.587, color="purple", linestyle=":", label="Phase 4B RF = 0.587")
ax.axvline(0.604, color="gray", linestyle=":", label="Majority = 0.604")
ax.set_xlabel("Test accuracy (per held-out subject)")
ax.set_ylabel("Number of subjects")
ax.set_title("Phase 5.4 — Distribution of LOSO accuracies")
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(out_dir, "accuracy_distribution.png"), dpi=150)
plt.close()
 
# Global confusion matrix ----------------------------------------
fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(cm_global, annot=True, fmt="d", cmap="Blues", xticklabels=["Relaxed", "Stress"], yticklabels=["Relaxed", "Stress"], ax=ax)
ax.set_xlabel("Predicted"); ax.set_ylabel("True")
ax.set_title(f"Phase 5.4 LOSO global confusion matrix\n"f"(2400 predictions across 40 folds; acc={global_acc:.3f})")
plt.tight_layout()
plt.savefig(os.path.join(out_dir, "global_confusion_matrix.png"), dpi=150)
plt.close()
 
# Final comparison table ----------------------------------------
comparison = {
    "Phase 4B RF (LOSO)":                  0.5870,
    "Phase 5.1  Simple CNN":               0.4833,
    "Phase 5.1b Simple CNN v2 (held-out)": 0.5500,
    "Phase 5.2  EEGNet (held-out)":        0.5167,
    "Phase 5.2b EEGNet + max-norm":        0.5250,
    "Phase 5.4  Simple CNN v2 LOSO":       float(accs.mean()),
}
 
# Comparison bar plot
fig, ax = plt.subplots(figsize=(10, 5))
names = list(comparison.keys())
values = list(comparison.values())
bar_colors = ["#777777"] + ["#F15854"] + ["#5DA5DA"] + ["#F15854"] * 2 + ["#60BD68"]
bars = ax.barh(names, values, color=bar_colors)
ax.axvline(0.604, color="gray", linestyle=":", label="Majority baseline (0.604)")
ax.set_xlabel("Accuracy"); ax.set_xlim(0.4, 0.7)
ax.set_title("Phase 5 — Final model comparison")
ax.legend(loc="lower right"); ax.grid(axis="x", alpha=0.3)
for bar, v in zip(bars, values):
    ax.text(v + 0.002, bar.get_y() + bar.get_height() / 2,
            f"{v:.3f}", va="center", fontsize=9)
plt.tight_layout()
plt.savefig(os.path.join(out_dir, "final_comparison.png"), dpi=150)
plt.close()
 
# JSON summary ----------------------------------------
summary = {
    "phase": "5.4 LOSO",
    "architecture": "SimpleEEGCNNv2",
    "n_folds": len(per_fold),
    "n_params": count_parameters(SimpleEEGCNNv2()),
    "total_time_min": float(total_time / 60),
    "hyperparameters": HYPERPARAMS,
    "aggregate": {
        "accuracy_mean": float(accs.mean()),
        "accuracy_std":  float(accs.std()),
        "accuracy_min":  float(accs.min()),
        "accuracy_max":  float(accs.max()),
        "accuracy_median": float(np.median(accs)),
        "f1_mean":         float(f1s.mean()),
        "f1_macro_mean":   float(f1ms.mean()),
        "global_accuracy_pooled": float(global_acc),
        "subjects_above_phase4b": int((accs > 0.587).sum()),
        "subjects_above_majority": int((accs > 0.604).sum()),
    },
    "global_confusion_matrix": cm_global.tolist(),
    "comparison": comparison,
}
with open(os.path.join(out_dir, "results.json"), "w") as f:
    json.dump(summary, f, indent=2)
 
print(f"\nSaved outputs to: {out_dir}")
print("  - per_fold_results.csv")
print("  - per_subject_accuracy.png")
print("  - accuracy_distribution.png")
print("  - global_confusion_matrix.png")
print("  - final_comparison.png")
print("  - results.json")
print("=" * 70)
print("Phase 5.4 complete")
print("=" * 70)


Saved outputs to: C:\Users\hibro\OneDrive\Desktop\Desktop_Files\Projects\Python\ML_Models\Cognitive_Stress_Classification\EEG-Stress-Classification\Results\phase5\phase5_4_loso
  - per_fold_results.csv
  - per_subject_accuracy.png
  - accuracy_distribution.png
  - global_confusion_matrix.png
  - final_comparison.png
  - results.json
Phase 5.4 complete
